# Reelcraft — générer un clip image → vidéo gratuitement

Ton portable a 2 Go de VRAM : bien trop peu pour ces modèles, qui en demandent 8 à 12.
Colab prête un GPU T4 de **16 Go**, gratuitement. On génère le clip ici, puis on
l'importe dans Reelcraft sur ta machine.

**Avant de commencer :** menu `Exécution` → `Modifier le type d'exécution` → **T4 GPU**.
Sans ça, rien ne fonctionnera.


## 1. Vérifier le GPU


In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit('Aucun GPU. Exécution > Modifier le type d'exécution > T4 GPU')

nom = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU  : {nom}')
print(f'VRAM : {vram:.1f} Go')
print()
print('Ton portable  : NVIDIA MX350, 2.0 Go')
print(f'Ici           : {vram:.0f} Go, soit {vram/2:.0f} fois plus')


## 2. Installer les dépendances

Deux à trois minutes. Les avertissements de version sont normaux.


In [ ]:
!pip install -q diffusers==0.31.0 transformers accelerate imageio imageio-ffmpeg
print('installe')


## 3. Accepter la licence du modèle

Stable Video Diffusion demande d'accepter ses conditions une seule fois :

1. Ouvre <https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt>
2. Connecte-toi et clique sur le bouton d'acceptation de la licence
3. Crée un jeton en lecture : <https://huggingface.co/settings/tokens>
4. Colle-le ci-dessous

Le jeton reste dans cette session Colab et n'est enregistré nulle part.


In [ ]:
from getpass import getpass
from huggingface_hub import login

login(getpass('Jeton Hugging Face (hf_...) : '))
print('connecte')


## 4. Téléverser ton image

Une photo nette, sujet bien visible. Le format portrait convient à du 9:16.


In [ ]:
from google.colab import files
from PIL import Image

envoi = files.upload()
chemin = next(iter(envoi))
image = Image.open(chemin).convert('RGB')
print(f'{chemin} : {image.size[0]}x{image.size[1]}')
image.resize((image.width // 3, image.height // 3))


## 5. Charger le modèle

Environ 5 Go à télécharger la première fois. `enable_model_cpu_offload` déplace les
poids entre le GPU et la RAM au fil des étapes : c'est plus lent, mais c'est ce qui
permet de tenir dans les 16 Go du T4.


In [ ]:
import torch
from diffusers import StableVideoDiffusionPipeline

pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16,
    variant='fp16',
)
pipe.enable_model_cpu_offload()
print('modele pret')


## 6. Générer

Compte 3 à 6 minutes sur un T4.

Le réglage qui compte est `motion_bucket_id` : il décide de l'ampleur du mouvement.
Autour de 60 le plan reste posé, au-delà de 180 la scène s'agite et se déforme.

**Une limite à connaître :** ce modèle ne lit aucun texte. Il déduit le mouvement de
l'image seule, on ne peut donc pas lui demander une action précise. Pour diriger la
scène par une phrase, il faut un modèle guidé par texte — voir la dernière section.


In [ ]:
import torch

# 576x1024 est la resolution d'entrainement du modele : s'en ecarter degrade nettement.
entree = image.resize((576, 1024))

generateur = torch.manual_seed(42)   # change la graine pour un autre resultat
frames = pipe(
    entree,
    decode_chunk_size=2,             # baisse a 1 en cas de saturation memoire
    generator=generateur,
    motion_bucket_id=110,            # 60 = discret, 180 = tres agite
    noise_aug_strength=0.05,
).frames[0]

print(f'{len(frames)} images generees')


## 7. Enregistrer et récupérer le MP4


In [ ]:
from diffusers.utils import export_to_video
from google.colab import files

sortie = 'motion-clip.mp4'
export_to_video(frames, sortie, fps=7)
files.download(sortie)
print(f'{sortie} pret au telechargement')


## 8. L'importer dans Reelcraft

Sur ta machine, l'API doit tourner. Puis :

```bash
cd d:/doss-M/vidproj

# lister les projets pour recuperer un id
backend/.venv/Scripts/python.exe scripts/import_motion_clip.py motion-clip.mp4

# rattacher le clip a la premiere scene du projet
backend/.venv/Scripts/python.exe scripts/import_motion_clip.py motion-clip.mp4 --project <id>
```

Ensuite, un rendu normal : Reelcraft compose les textes, la musique et la voix off
par-dessus le clip, exactement comme si l'AI Motion payante l'avait produit.


## Si tu veux diriger le mouvement par une phrase

Stable Video Diffusion ignore le texte. Pour qu'une consigne écrite guide la scène,
remplace les cellules 5 et 6 par un modèle guidé par texte :

| Modèle | Taille | Tient sur un T4 |
|---|---|---|
| `Wan-AI/Wan2.2-TI2V-5B-Diffusers` | 5 Md | oui, tout juste |
| `Lightricks/LTX-Video` | 2 Md | oui, confortable |

Les deux acceptent un `prompt`. Dis-le-moi et je te prépare la variante.
